<div style="background:linear-gradient(120deg,#1f1f3a 0%,#3b2e7e 55%,#0f6cbd 100%);background-color:#3b2e7e;border-radius:12px;padding:22px 28px;color:#ffffff"><table style="border:none;border-collapse:collapse;width:100%;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0;vertical-align:middle;text-align:left;color:#ffffff"><div style="font-size:13px;letter-spacing:2px;text-transform:uppercase;opacity:0.85">Microsoft Fabric &middot; Lakehouse &middot; Component</div><div style="font-size:34px;font-weight:700;line-height:1.2;margin:2px 0 6px 0">Backup and Restore Engine</div><div style="font-size:15px;line-height:1.5;max-width:520px">The functions that copy a lakehouse (Delta tables and Files) to a write-once ADLS Gen2 container, check the copy, and bring it back. The Backup Utility and Restore Utility notebooks both run on this.</div></td><td style="border:none;padding:0 0 0 20px;vertical-align:middle;text-align:right"></td></tr></table></div><div style="margin:12px 2px 0 2px;font-size:14px;line-height:1.6"><b>This notebook is not opened to do a job.</b> It only defines functions and copies nothing when run. It is loaded with <code>%run nb_backup_restore_adls_v6</code> by the Backup Utility and Restore Utility notebooks.<br/><span style="display:inline-block;background:#107c10;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Defines functions only</span> &nbsp; <span style="display:inline-block;background:#5c2e91;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Change with care: both utilities depend on it</span></div><div style="margin:12px 2px 0 2px;font-size:14px;line-height:1.55"><b>Rules the code follows</b>, each found by testing on the write-once container:<ul style="margin:6px 0 6px 0"><li style="margin:2px 0">Everything written to the archive goes through <code>fastcp</code>. <code>fs.put</code> and <code>fs.cp</code> fail there and leave empty files that cannot be deleted.</li><li style="margin:2px 0">Tables are copied <b>without</b> their <code>_metadata</code> folder; with it, the copy cannot be queried through a shortcut. Table history (time travel) is in <code>_delta_log</code> and is kept.</li><li style="margin:2px 0">The manifest is written last. A backup folder without a manifest is incomplete and is ignored by restore.</li><li style="margin:2px 0">Nothing in the archive is ever overwritten or deleted. A failed run stays until retention ends; the next run uses a new timestamp.</li></ul></div><div style="margin:8px 2px 0 2px;font-size:13px;line-height:1.7"><b>Where a backup lives</b><br/><code>&lt;archive&gt;/backups/&lt;timestamp&gt;/Tables/&lt;schema&gt;/&lt;table&gt;/</code> &nbsp; one Delta table each, no <code>_metadata</code><br/><code>&lt;archive&gt;/backups/&lt;timestamp&gt;/Files/...</code> &nbsp; only when Files were included<br/><code>&lt;archive&gt;/backups/&lt;timestamp&gt;/_manifest/manifest.json</code> &nbsp; the completion marker</div><div style="margin:10px 2px 0 2px;font-size:13px;line-height:1.55;opacity:0.9"><b>Not covered:</b> shortcuts in the source (skipped and listed in the manifest), anything outside Tables and Files, permissions, security roles, SQL endpoint views.</div>


<div style="border-left:6px solid #3b3a39;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#3b3a39;font-weight:700">Part 1</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Setup and shared helpers</div><div style="font-size:14px;line-height:1.55">Imports, the logger, the OneLake address for this tenant, and the small functions everything else uses.<ul style="margin:6px 0 6px 0"><li style="margin:2px 0"><code>lakehouse_root</code>: the OneLake address of a lakehouse from its workspace ID and lakehouse ID.</li><li style="margin:2px 0"><code>lakehouse_info</code>: workspace name, lakehouse name and whether the lakehouse uses schemas.</li><li style="margin:2px 0"><code>shortcut_paths</code>: every shortcut in a lakehouse, so shortcuts are never copied or deleted.</li><li style="margin:2px 0"><code>delta_state</code>: version, file count and size of a Delta table: the fingerprint used to compare source and copy.</li><li style="margin:2px 0"><code>exclude_flag_supported</code>: tests, inside the staging lakehouse, whether <code>fastcp</code> can skip <code>_metadata</code> directly.</li><li style="margin:2px 0"><code>copy_table_without_metadata</code>: copies one table to the archive without <code>_metadata</code>, directly or through staging, and stops if <code>_metadata</code> ever reaches the archive.</li></ul><b>Expect:</b> no output.</div></td></tr></table></div>


In [ ]:
import json, logging, os, re, shutil, sys, time, uuid
from datetime import datetime, timezone

import requests
import notebookutils
from notebookutils import fs
from delta.tables import DeltaTable

logger = logging.getLogger("LakehouseArchive")
logger.setLevel(logging.INFO)
logger.propagate = False
if not logger.handlers:
    _h = logging.StreamHandler(sys.stdout)
    _h.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    logger.addHandler(_h)

FABRIC_API = "https://api.fabric.microsoft.com/v1/"
ONELAKE = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "").rstrip("/")
EXCLUDE_METADATA = {"flags": "--exclude-path=_metadata"}


def lakehouse_root(workspace_id, lakehouse_id):
    return f"abfss://{workspace_id}@{ONELAKE}/{lakehouse_id}"


def normalize_archive_uri(uri):
    uri = str(uri).strip().rstrip("/")
    if not re.match(r"^abfss://[^@/]+@[^/]+\.dfs\.core\.windows\.net(/.*)?$", uri):
        raise ValueError(
            "archive_root_uri must look like "
            "abfss://<container>@<account>.dfs.core.windows.net[/<folder>]"
        )
    return uri


def check_timestamp(value):
    value = str(value)
    if not re.fullmatch(r"\d{12}", value):
        raise ValueError(f"backup timestamp must be 12 digits (YYYYMMDDHHMM), got {value!r}")
    return value


def q(name):
    return "`" + str(name).replace("`", "``") + "`"


def reason(exc):
    lines = [l.strip() for l in str(exc).splitlines() if l.strip() and not l.strip().startswith("at ")]
    return " | ".join(lines[:3])[:600]


def fabric_get(path, params=None, attempts=5):
    headers = {"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi")}
    for attempt in range(1, attempts + 1):
        response = requests.get(FABRIC_API + path, headers=headers, params=params, timeout=60)
        if response.status_code in (429, 500, 502, 503, 504) and attempt < attempts:
            time.sleep(int(response.headers.get("Retry-After", 2 ** attempt)))
            continue
        response.raise_for_status()
        return response.json()


def lakehouse_info(workspace_id, lakehouse_id):
    item = fabric_get(f"workspaces/{workspace_id}/lakehouses/{lakehouse_id}")
    workspace = fabric_get(f"workspaces/{workspace_id}")
    return {
        "workspace_name": workspace["displayName"],
        "lakehouse_name": item["displayName"],
        "has_schemas": bool((item.get("properties") or {}).get("defaultSchema")),
    }


def shortcut_paths(workspace_id, lakehouse_id):
    """Relative paths of every shortcut in the lakehouse, e.g. 'Tables/dbo/award'."""
    found, token = set(), None
    while True:
        page = fabric_get(
            f"workspaces/{workspace_id}/items/{lakehouse_id}/shortcuts",
            {"continuationToken": token} if token else None,
        )
        for s in page.get("value", []):
            found.add(f"{s['path'].strip('/')}/{s['name']}")
        token = page.get("continuationToken")
        if not token:
            return found


def child_dirs(path):
    return sorted(i.name.rstrip("/") for i in fs.ls(path) if i.isDir)


def delta_state(path):
    table = DeltaTable.forPath(spark, path)
    detail = table.detail().first()
    return {
        "version": int(table.history(1).first()["version"]),
        "numFiles": int(detail["numFiles"]),
        "sizeInBytes": int(detail["sizeInBytes"]),
    }


def read_json(path):
    return json.loads(spark.read.text(path, wholetext=True).first()[0])


def run_root_of(archive_root_uri, backup_timestamp):
    return f"{normalize_archive_uri(archive_root_uri)}/backups/{check_timestamp(backup_timestamp)}"


def manifest_path_of(archive_root_uri, backup_timestamp):
    return run_root_of(archive_root_uri, backup_timestamp) + "/_manifest/manifest.json"


def exclude_flag_supported(stage_root):
    """Checks, inside the staging lakehouse, whether fastcp can skip _metadata.
    Done in staging because a wrong copy there can be deleted; in the archive it cannot."""
    probe = f"{stage_root}/_probe_{uuid.uuid4().hex[:8]}"
    try:
        fs.put(f"{probe}/src/t/_metadata/x.txt", "x", True)
        fs.put(f"{probe}/src/t/_delta_log/y.txt", "y", True)
        fs.mkdirs(f"{probe}/dst")
        fs.fastcp(f"{probe}/src/t", f"{probe}/dst", True, extraConfigs=EXCLUDE_METADATA)
        return fs.exists(f"{probe}/dst/t/_delta_log/y.txt") and not fs.exists(f"{probe}/dst/t/_metadata")
    except Exception as exc:
        logger.info("fastcp skip option not usable: %s", reason(exc))
        return False
    finally:
        try:
            fs.rm(probe, True)
        except Exception:
            pass


def copy_table_without_metadata(source_path, dest_parent, table, stage_root, use_flag):
    """Copies one Delta table folder into dest_parent, leaving _metadata behind."""
    dest = f"{dest_parent}/{table}"
    if use_flag:
        fs.fastcp(source_path, dest_parent, True, extraConfigs=EXCLUDE_METADATA)
    else:
        stage_parent = f"{stage_root}/{uuid.uuid4().hex[:8]}"
        try:
            fs.mkdirs(stage_parent)
            fs.fastcp(source_path, stage_parent, True)
            staged = f"{stage_parent}/{table}"
            if fs.exists(staged + "/_metadata"):
                fs.rm(staged + "/_metadata", True)      # staging copy only
            fs.fastcp(staged, dest_parent, True)
        finally:
            try:
                fs.rm(stage_parent, True)
            except Exception:
                pass
    if not fs.exists(dest + "/_delta_log"):
        raise RuntimeError(f"Copy failed, no _delta_log at {dest}")
    if fs.exists(dest + "/_metadata"):
        raise RuntimeError(
            f"_metadata reached the archive at {dest}. It cannot be removed. "
            "Stop, and rerun with a new timestamp after fixing the copy method."
        )
    return dest

<div style="border-left:6px solid #c4314b;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#c4314b;font-weight:700">Part 2</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Backup</div><div style="font-size:14px;line-height:1.55"><span style="display:inline-block;background:#c4314b;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">backup_lakehouse_to_archive writes to the archive</span><br/>One function: <code>backup_lakehouse_to_archive(source workspace, source lakehouse, archive, staging workspace, staging lakehouse, ...)</code>. It returns the backup timestamp.<ol style="margin:6px 0 6px 0"><li style="margin:2px 0"><b>Checks first, writes nothing:</b> the archive is reachable, the timestamp is unused, the staging lakehouse is writable and is not the source.</li><li style="margin:2px 0"><b>Plans:</b> lists every Delta table; shortcuts and non-Delta folders are skipped and recorded.</li><li style="margin:2px 0"><b>Copies each table,</b> then compares version, file count and size before the copy, after it, and on the copy. With <code>verify_counts</code> it also compares row counts.</li><li style="margin:2px 0"><b>Copies Files</b> when <code>copy_files</code> is set.</li><li style="margin:2px 0"><b>Writes the manifest last</b> and reads it back.</li></ol><b>Ends with one line:</b> <code>BACKUP COMPLETE: &lt;timestamp&gt; | N tables | ...</code> or <code>BACKUP FAILED: ...</code> with the reason. A table that changes during its copy fails the run, which is why nothing should be writing to the source.</div></td></tr></table></div>


In [ ]:
def backup_lakehouse_to_archive(
    src_ws_id, src_lh_id,
    archive_root_uri,
    staging_ws_id, staging_lh_id,
    backup_timestamp=None,
    copy_files=False,
    verify_counts=False,
    schemas=None,
    tables=None,
):
    """Copies every Delta table (and optionally Files) of a lakehouse to the archive.

    staging_*   a lakehouse used for a short capability check and, if fastcp cannot
                skip _metadata, as a temporary stop for each table. Must not be the source.
    schemas     optional set of schema names to include.
    tables      optional set of table names to include.
    Returns the backup timestamp."""
    if not isinstance(copy_files, bool) or not isinstance(verify_counts, bool):
        raise TypeError("copy_files and verify_counts must be True or False")
    if (staging_ws_id, staging_lh_id) == (src_ws_id, src_lh_id):
        raise ValueError("The staging lakehouse must not be the source lakehouse")

    archive = normalize_archive_uri(archive_root_uri)
    ts = check_timestamp(backup_timestamp or datetime.now(timezone.utc).strftime("%Y%m%d%H%M"))
    run_root = f"{archive}/backups/{ts}"
    source_root = lakehouse_root(src_ws_id, src_lh_id)
    stage_root = f"{lakehouse_root(staging_ws_id, staging_lh_id)}/Files/backup_staging/{ts}"

    # ---- checks before anything is written -------------------------------
    try:
        if fs.exists(run_root):
            raise RuntimeError(f"{run_root} already exists. Use a new timestamp.")
    except RuntimeError:
        raise
    except Exception as exc:
        raise RuntimeError(f"Cannot reach the archive {archive}: {reason(exc)}")

    info = lakehouse_info(src_ws_id, src_lh_id)
    shortcuts = shortcut_paths(src_ws_id, src_lh_id)
    tables_root = f"{source_root}/Tables"
    plan, skipped_shortcuts, skipped_non_delta = [], [], []

    def consider(schema, table, rel, path):
        if tables is not None and table not in tables:
            return
        if rel in shortcuts:
            skipped_shortcuts.append(rel)
        elif not fs.exists(path + "/_delta_log"):
            skipped_non_delta.append(rel)
        else:
            plan.append((schema, table, path))

    if info["has_schemas"]:
        for schema in child_dirs(tables_root):
            if schemas is not None and schema not in schemas:
                continue
            if f"Tables/{schema}" in shortcuts:
                skipped_shortcuts.append(f"Tables/{schema}")
                continue
            for table in child_dirs(f"{tables_root}/{schema}"):
                consider(schema, table, f"Tables/{schema}/{table}", f"{tables_root}/{schema}/{table}")
    else:
        for table in child_dirs(tables_root):
            consider("dbo", table, f"Tables/{table}", f"{tables_root}/{table}")

    if not plan:
        raise RuntimeError("Nothing to back up: no Delta tables matched.")
    logger.info("Backup %s: %d tables planned, %d shortcuts skipped, %d non-Delta folders skipped",
                ts, len(plan), len(skipped_shortcuts), len(skipped_non_delta))

    try:
        fs.mkdirs(stage_root)
    except Exception as exc:
        raise RuntimeError(
            "Cannot write to the staging lakehouse. Check that staging_ws_id is the workspace "
            f"that holds staging_lh_id. Nothing was written to the archive. {reason(exc)}"
        )
    use_flag = exclude_flag_supported(stage_root)
    logger.info("Copy method: %s", "fastcp skipping _metadata" if use_flag else "staged copy")

    # ---- tables ----------------------------------------------------------
    started = datetime.now(timezone.utc)
    done, made_parents = [], set()
    try:
        for n, (schema, table, source_path) in enumerate(plan, 1):
            dest_parent = f"{run_root}/Tables/{schema}"
            if dest_parent not in made_parents:
                fs.mkdirs(dest_parent)
                made_parents.add(dest_parent)

            before = delta_state(source_path)
            dest = copy_table_without_metadata(source_path, dest_parent, table, stage_root, use_flag)
            after = delta_state(source_path)
            copied = delta_state(dest)
            if after != before:
                raise RuntimeError(f"{schema}.{table} changed during the copy: {before} -> {after}")
            if copied != before:
                raise RuntimeError(f"{schema}.{table} copy differs from source: {copied} != {before}")

            entry = {"schema": schema, "table": table, **before}
            if verify_counts:
                rows_src = spark.read.format("delta").load(source_path).count()
                rows_dst = spark.read.format("delta").load(dest).count()
                if rows_src != rows_dst:
                    raise RuntimeError(f"{schema}.{table} row count differs: {rows_dst} != {rows_src}")
                entry["rows"] = rows_dst
            done.append(entry)
            logger.info("[%d/%d] %s.%s copied (version %d, %d files)",
                        n, len(plan), schema, table, before["version"], before["numFiles"])

        # ---- files -------------------------------------------------------
        files_copied = False
        if copy_files:
            source_files = f"{source_root}/Files"
            if fs.ls(source_files):
                fs.fastcp(source_files, run_root, True)          # creates <run_root>/Files
                if not fs.exists(f"{run_root}/Files"):
                    raise RuntimeError("Files copy failed")
                files_copied = True
                logger.info("Files copied")
            else:
                logger.info("Files is empty, nothing to copy")

        # ---- manifest, written last -------------------------------------
        manifest = {
            "status": "complete",
            "backup_timestamp": ts,
            "started_utc": started.isoformat(),
            "completed_utc": datetime.now(timezone.utc).isoformat(),
            "source": {
                "workspace_id": src_ws_id, "lakehouse_id": src_lh_id,
                "workspace_name": info["workspace_name"], "lakehouse_name": info["lakehouse_name"],
                "has_schemas": info["has_schemas"],
            },
            "archive_run_root": run_root,
            "copy_method": "fastcp_exclude" if use_flag else "staged",
            "files_copied": files_copied,
            "verify_counts": verify_counts,
            "tables": done,
            "skipped_shortcuts": sorted(skipped_shortcuts),
            "skipped_non_delta": sorted(skipped_non_delta),
        }
        local_parent = f"/tmp/lakehouse_backup_{ts}_{uuid.uuid4().hex[:6]}"
        local_dir = f"{local_parent}/_manifest"
        os.makedirs(local_dir, exist_ok=True)
        try:
            with open(f"{local_dir}/manifest.json", "w") as handle:
                json.dump(manifest, handle, indent=2)
            fs.fastcp(f"file:{local_dir}", run_root, True)        # never fs.put on the archive
        finally:
            shutil.rmtree(local_parent, ignore_errors=True)

        written = read_json(f"{run_root}/_manifest/manifest.json")
        if written.get("status") != "complete" or len(written.get("tables", [])) != len(done):
            raise RuntimeError("Manifest in the archive does not match what was written")
    except Exception as exc:
        logger.error("Backup %s FAILED after %d of %d tables. The run folder has no manifest, "
                     "so it counts as incomplete. Rerun with a new timestamp.", ts, len(done), len(plan))
        print(f"BACKUP FAILED: {ts} | stopped after {len(done)} of {len(plan)} tables | {reason(exc)}")
        raise
    finally:
        try:
            if fs.exists(stage_root):
                fs.rm(stage_root, True)
        except Exception:
            pass

    logger.info("Backup %s complete: %d tables at %s", ts, len(done), run_root)
    minutes = (datetime.now(timezone.utc) - started).total_seconds() / 60
    print(f"BACKUP COMPLETE: {ts} | {len(done)} tables | "
          f"Files {'included' if files_copied else 'not included'} | {minutes:.1f} min")
    return ts

<div style="border-left:6px solid #107c10;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#107c10;font-weight:700">Part 3</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">List and verify</div><div style="font-size:14px;line-height:1.55"><span style="display:inline-block;background:#107c10;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Read-only</span><ul style="margin:6px 0 6px 0"><li style="margin:2px 0"><code>list_archive_backups</code>: every backup folder in an archive and whether it is complete.</li><li style="margin:2px 0"><code>latest_complete_backup</code>: the timestamp of the newest complete backup.</li><li style="margin:2px 0"><code>verify_archive_backup</code>: re-reads every table of a backup from the archive and compares it with the manifest. One row per table, <code>OK</code> or <code>FAIL</code> with the reason.</li></ul><code>verify_archive_backup</code> also prints the folder and name to use if someone wants to create shortcuts by hand to query a backup. Those go on the top-level Tables node of a schema-enabled lakehouse, not inside <code>dbo</code>.</div></td></tr></table></div>


In [ ]:
def list_archive_backups(archive_root_uri):
    """Every run folder in the archive and whether it has a manifest."""
    archive = normalize_archive_uri(archive_root_uri)
    rows = []
    for ts in child_dirs(f"{archive}/backups"):
        complete = fs.exists(f"{archive}/backups/{ts}/_manifest/manifest.json")
        rows.append({"backup_timestamp": ts, "complete": complete})
    return rows


def verify_archive_backup(archive_root_uri, backup_timestamp):
    """Re-reads every table in a backup and compares it with the manifest."""
    run_root = run_root_of(archive_root_uri, backup_timestamp)
    manifest = read_json(manifest_path_of(archive_root_uri, backup_timestamp))
    rows = []
    for t in manifest["tables"]:
        path = f"{run_root}/Tables/{t['schema']}/{t['table']}"
        expected = {k: t[k] for k in ("version", "numFiles", "sizeInBytes")}
        try:
            actual = delta_state(path)
            if fs.exists(path + "/_metadata"):
                status = "FAIL: _metadata present"
            elif actual != expected:
                status = f"FAIL: {actual} != {expected}"
            else:
                status = "OK"
        except Exception as exc:
            status = "FAIL: " + reason(exc)
        rows.append({"schema": t["schema"], "table": t["table"], "version": t["version"], "status": status})

    bad = [r for r in rows if r["status"] != "OK"]
    print(f"Backup {backup_timestamp}: {len(rows) - len(bad)} of {len(rows)} tables OK")
    print("\nTo query this backup by name, create one schema shortcut by hand per line below,")
    print("on the top-level Tables node of a schema-enabled lakehouse (not inside dbo):")
    for schema in sorted({t["schema"] for t in manifest["tables"]}):
        print(f"  name {schema}_{backup_timestamp}  ->  {run_root}/Tables/{schema}")
    return rows


def latest_complete_backup(archive_root_uri):
    """Timestamp of the newest backup that has a manifest."""
    done = [r["backup_timestamp"] for r in list_archive_backups(archive_root_uri) if r["complete"]]
    if not done:
        raise RuntimeError(f"No complete backup found under {normalize_archive_uri(archive_root_uri)}/backups")
    return max(done)

<div style="border-left:6px solid #0e7a6e;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#0e7a6e;font-weight:700">Part 4</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Restore, preview and cleanup</div><div style="font-size:14px;line-height:1.55"><span style="display:inline-block;background:#c4314b;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">restore_from_archive writes to the target</span> &nbsp; <span style="display:inline-block;background:#8a6d00;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">clean_target_for_restore deletes from the target</span><ul style="margin:6px 0 6px 0"><li style="margin:2px 0"><code>preview_restore</code>: prints what a restore would do and changes nothing. Returns what is already in the way.</li><li style="margin:2px 0"><code>restore_from_archive</code>: copies tables from the archive into a lakehouse, compares each with the manifest and registers it. By default it never overwrites: if any destination exists, nothing is restored.</li><li style="margin:2px 0"><code>restore_files_from_archive</code>: copies the backup&#39;s Files straight under the target&#39;s <code>Files</code> folder; refuses if any top-level name already exists.</li><li style="margin:2px 0"><code>clean_target_for_restore</code>: deletes from the target so a restore can run. <code>replace</code> removes only what the backup puts back; <code>all</code> removes every table and file.</li></ul><b>Safeguards on cleanup:</b> the target lakehouse name must be passed exactly; the backup must be complete and pass verification before anything is deleted; shortcuts are never deleted, and a shortcut sitting where the backup needs to restore stops the run first.<br/><b>Ends with one line:</b> <code>RESTORE COMPLETE: ...</code> or <code>RESTORE FINISHED WITH PROBLEMS: ...</code>.</div></td></tr></table></div>


In [ ]:
def restore_files_from_archive(archive_root_uri, backup_timestamp, tgt_ws_id, tgt_lh_id, check_only=False):
    """Copies everything under the backup's Files folder straight into the target's Files folder.
    Never overwrites: if any top-level name already exists in the target's Files, nothing is copied.
    check_only=True runs the checks and copies nothing. Returns the names restored (or to restore)."""
    source_files = run_root_of(archive_root_uri, backup_timestamp) + "/Files"
    target_files = lakehouse_root(tgt_ws_id, tgt_lh_id) + "/Files"
    if not fs.exists(source_files):
        logger.info("This backup holds no Files")
        return []
    items = [(i.name.rstrip("/"), i.isDir) for i in fs.ls(source_files)]
    clashes = [name for name, _ in items if fs.exists(f"{target_files}/{name}")]
    if clashes:
        raise RuntimeError(f"Already present in the target's Files; nothing was copied: {sorted(clashes)}")
    if check_only:
        return [name for name, _ in items]
    for name, is_dir in items:
        if is_dir:
            fs.fastcp(f"{source_files}/{name}", target_files, True)
        else:
            fs.cp(f"{source_files}/{name}", f"{target_files}/{name}")
        if not fs.exists(f"{target_files}/{name}"):
            raise RuntimeError(f"Files restore failed at {name}")
        logger.info("Files/%s restored", name)
    return [name for name, _ in items]


def restore_from_archive(
    archive_root_uri, backup_timestamp,
    tgt_ws_id, tgt_lh_id,
    schemas=None,
    tables=None,
    restore_files=False,
    allow_unverified=False,
    clean_target=False,
    confirm_target_name=None,
):
    """Copies tables from the archive into a lakehouse and registers them.

    Without clean_target it never overwrites: every destination table folder must be absent.
    schemas / tables   optional sets to restore only part of a backup.
    allow_unverified   restore a run that has no manifest (an incomplete backup).
    clean_target       False (default), "replace" or "all": delete from the target first.
                       See clean_target_for_restore. Needs confirm_target_name."""
    if clean_target and allow_unverified:
        raise RuntimeError("clean_target cannot be used with allow_unverified")
    run_root = run_root_of(archive_root_uri, backup_timestamp)
    ts = check_timestamp(backup_timestamp)
    manifest_path = manifest_path_of(archive_root_uri, ts)

    if fs.exists(manifest_path):
        manifest = read_json(manifest_path)
        entries = [(t["schema"], t["table"], {k: t[k] for k in ("version", "numFiles", "sizeInBytes")})
                   for t in manifest["tables"]]
    elif allow_unverified:
        logger.warning("No manifest for %s. Restoring whatever is in the run folder.", ts)
        entries = [(s, t, None)
                   for s in child_dirs(f"{run_root}/Tables")
                   for t in child_dirs(f"{run_root}/Tables/{s}")
                   if fs.exists(f"{run_root}/Tables/{s}/{t}/_delta_log")]
    else:
        raise RuntimeError(f"No manifest for {ts}: the backup is incomplete. "
                           "Pass allow_unverified=True to restore it anyway.")

    entries = [e for e in entries
               if (schemas is None or e[0] in schemas) and (tables is None or e[1] in tables)]
    if not entries:
        raise RuntimeError("Nothing to restore: no tables matched.")

    info = lakehouse_info(tgt_ws_id, tgt_lh_id)
    target_root = lakehouse_root(tgt_ws_id, tgt_lh_id)

    # ---- plan and check before anything is written -----------------------
    plan = []
    for schema, table, expected in entries:
        parent = f"{target_root}/Tables/{schema}" if info["has_schemas"] else f"{target_root}/Tables"
        plan.append((schema, table, expected, f"{run_root}/Tables/{schema}/{table}", parent))
    if not info["has_schemas"]:
        names = [p[1] for p in plan]
        clashes = sorted({n for n in names if names.count(n) > 1})
        if clashes:
            raise RuntimeError(f"Target has no schemas and these table names repeat across schemas: {clashes}")
    if clean_target:
        clean_target_for_restore(
            archive_root_uri, ts, tgt_ws_id, tgt_lh_id,
            mode=clean_target, confirm_target_name=confirm_target_name,
            include_files=restore_files, schemas=schemas, tables=tables,
        )
    existing = [f"{p[4]}/{p[1]}" for p in plan if fs.exists(f"{p[4]}/{p[1]}")]
    if existing:
        raise RuntimeError("These destination folders already exist; nothing was restored:\n  "
                           + "\n  ".join(existing))

    if restore_files:
        restore_files_from_archive(archive_root_uri, ts, tgt_ws_id, tgt_lh_id, check_only=True)

    # ---- copy and register ----------------------------------------------
    catalog = f"{q(info['workspace_name'])}.{q(info['lakehouse_name'])}"
    results, made = [], set()
    for n, (schema, table, expected, source_path, parent) in enumerate(plan, 1):
        dest = f"{parent}/{table}"
        if parent not in made:
            if not fs.exists(parent):
                fs.mkdirs(parent)
            if info["has_schemas"]:
                spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{q(schema)}")
            made.add(parent)

        fs.fastcp(source_path, parent, True)
        if not fs.exists(dest + "/_delta_log"):
            raise RuntimeError(f"Copy failed, no _delta_log at {dest}")
        actual = delta_state(dest)
        if expected is not None and actual != expected:
            raise RuntimeError(f"{schema}.{table} restored copy differs from manifest: {actual} != {expected}")

        identifier = (f"{catalog}.{q(schema)}.{q(table)}" if info["has_schemas"]
                      else f"{q(info['lakehouse_name'])}.{q(table)}")
        status = "restored"
        try:
            spark.sql(f"CREATE TABLE IF NOT EXISTS {identifier} USING DELTA LOCATION '{dest}'")
            spark.sql(f"SELECT 1 FROM {identifier} LIMIT 1").collect()
        except Exception as exc:
            status = "copied, registration failed: " + reason(exc)
            logger.warning("%s.%s %s", schema, table, status)
        results.append({"schema": schema, "table": table, "version": actual["version"],
                        "status": status, "location": dest})
        logger.info("[%d/%d] %s.%s %s", n, len(plan), schema, table, status.split(":")[0])

    if restore_files:
        restore_files_from_archive(archive_root_uri, ts, tgt_ws_id, tgt_lh_id)

    logger.info("Restore of %s finished: %d tables", ts, len(results))
    problems = [r for r in results if r["status"] != "restored"]
    print(f"RESTORE {'COMPLETE' if not problems else 'FINISHED WITH PROBLEMS'}: {ts} | "
          f"{len(results) - len(problems)} of {len(results)} tables restored | "
          f"Files {'restored' if restore_files else 'not requested'}")
    return results


def preview_restore(
    archive_root_uri, backup_timestamp,
    tgt_ws_id, tgt_lh_id,
    schemas=None,
    tables=None,
    restore_files=False,
    clean_target=False,
):
    """Prints what a restore would do and changes nothing. Returns the list of clashes:
    tables and top-level Files names that already exist in the target."""
    ts = check_timestamp(backup_timestamp)
    run_root = run_root_of(archive_root_uri, ts)
    if not fs.exists(manifest_path_of(archive_root_uri, ts)):
        raise RuntimeError(f"No manifest for {ts}: the backup is incomplete. Choose another backup.")
    manifest = read_json(manifest_path_of(archive_root_uri, ts))
    info = lakehouse_info(tgt_ws_id, tgt_lh_id)
    target_root = lakehouse_root(tgt_ws_id, tgt_lh_id)

    chosen = [t for t in manifest["tables"]
              if (schemas is None or t["schema"] in schemas) and (tables is None or t["table"] in tables)]
    def rel(t):
        return f"Tables/{t['schema']}/{t['table']}" if info["has_schemas"] else f"Tables/{t['table']}"
    clashes = [{"path": rel(t), "kind": "table"} for t in chosen if fs.exists(f"{target_root}/{rel(t)}")]
    file_names = []
    if restore_files and fs.exists(f"{run_root}/Files"):
        file_names = [i.name.rstrip("/") for i in fs.ls(f"{run_root}/Files")]
        clashes += [{"path": f"Files/{n}", "kind": "files"} for n in file_names
                    if fs.exists(f"{target_root}/Files/{n}")]

    src = manifest["source"]
    print(f"Backup            : {ts}  (completed {manifest['completed_utc'][:16]} UTC)")
    print(f"Taken from        : {src['workspace_name']} / {src['lakehouse_name']}")
    print(f"Restore into      : {info['workspace_name']} / {info['lakehouse_name']}")
    print(f"Tables to restore : {len(chosen)} of {len(manifest['tables'])} in the backup")
    if not restore_files:
        print("Files             : not requested")
    elif not manifest.get("files_copied"):
        print("Files             : this backup holds no Files")
    else:
        print(f"Files             : {len(file_names)} top-level items -> {sorted(file_names)}")
    print(f"Already in target : {len(clashes)} items")
    if not clashes:
        print("RESULT            : nothing is in the way. The restore can run without cleanup.")
    elif not clean_target:
        print("RESULT            : the restore will STOP and change nothing, because the items below exist.")
        print("                    Use an empty target, or set CLEAN_TARGET and review again.")
    elif clean_target == "replace":
        print("RESULT            : the items below will be DELETED from the target, then restored.")
    else:
        print('RESULT            : CLEAN_TARGET is "all": EVERY table in the target (and everything under')
        print("                    Files, if Files are restored) will be DELETED, then the backup restored.")
    return clashes


def clean_target_for_restore(
    archive_root_uri, backup_timestamp,
    tgt_ws_id, tgt_lh_id,
    mode, confirm_target_name,
    include_files=False,
    schemas=None,
    tables=None,
    dry_run=False,
):
    """Deletes tables (and optionally Files content) from the target lakehouse so a restore can run.

    mode "replace"   delete only what the backup will put back: the tables in the backup and,
                     with include_files, the top-level Files names the backup holds.
    mode "all"       delete every table in the target and, with include_files, everything under Files.
    confirm_target_name   must equal the target lakehouse's name exactly, or nothing happens.
    dry_run=True     list what would be deleted and delete nothing.

    Safeguards: the backup must be complete and pass verification before anything is deleted;
    shortcuts are never deleted; if a shortcut sits where the backup needs to restore, it stops
    before deleting anything."""
    if mode not in ("replace", "all"):
        raise ValueError('mode must be "replace" or "all"')
    if mode == "all" and (schemas is not None or tables is not None):
        raise ValueError('mode "all" cannot be combined with schemas= or tables=; use "replace"')

    ts = check_timestamp(backup_timestamp)
    run_root = run_root_of(archive_root_uri, ts)
    info = lakehouse_info(tgt_ws_id, tgt_lh_id)
    if confirm_target_name != info["lakehouse_name"]:
        raise RuntimeError(
            f"Cleanup not confirmed. The target lakehouse is {info['lakehouse_name']!r}; "
            "pass confirm_target_name with exactly that name to allow deleting from it."
        )
    if not fs.exists(manifest_path_of(archive_root_uri, ts)):
        raise RuntimeError(f"Backup {ts} has no manifest. Cleanup is only allowed for a complete backup.")
    manifest = read_json(manifest_path_of(archive_root_uri, ts))

    target_root = lakehouse_root(tgt_ws_id, tgt_lh_id)
    has_schemas = info["has_schemas"]
    catalog = f"{q(info['workspace_name'])}.{q(info['lakehouse_name'])}"
    shortcuts = shortcut_paths(tgt_ws_id, tgt_lh_id)

    def table_rel(schema, table):
        return f"Tables/{schema}/{table}" if has_schemas else f"Tables/{table}"

    def touches_shortcut(rel):
        return any(rel == sc or rel.startswith(sc + "/") or sc.startswith(rel + "/") for sc in shortcuts)

    # what the restore will write
    incoming = [("table", table_rel(t["schema"], t["table"]), t["schema"], t["table"])
                for t in manifest["tables"]
                if (schemas is None or t["schema"] in schemas) and (tables is None or t["table"] in tables)]
    if include_files and fs.exists(f"{run_root}/Files"):
        incoming += [("files", f"Files/{i.name.rstrip('/')}", None, i.name.rstrip("/"))
                     for i in fs.ls(f"{run_root}/Files")]

    # what to delete
    if mode == "replace":
        wanted = list(incoming)
    else:
        wanted = []
        if has_schemas:
            for s in child_dirs(f"{target_root}/Tables"):
                if f"Tables/{s}" in shortcuts:
                    continue
                wanted += [("table", f"Tables/{s}/{t}", s, t) for t in child_dirs(f"{target_root}/Tables/{s}")]
        else:
            wanted += [("table", f"Tables/{t}", "dbo", t) for t in child_dirs(f"{target_root}/Tables")]
        if include_files:
            wanted += [("files", f"Files/{i.name.rstrip('/')}", None, i.name.rstrip("/"))
                       for i in fs.ls(f"{target_root}/Files")]

    blocked = sorted(rel for _, rel, _, _ in incoming
                     if touches_shortcut(rel) and fs.exists(f"{target_root}/{rel}"))
    if blocked:
        raise RuntimeError("A shortcut sits where the backup needs to restore. Remove it by hand first; "
                           "nothing was deleted:\n  " + "\n  ".join(blocked))

    todo, rows = [], []
    for kind, rel, schema, name in wanted:
        if not fs.exists(f"{target_root}/{rel}"):
            continue
        if touches_shortcut(rel):
            rows.append({"path": rel, "action": "kept (shortcut)"})
        else:
            todo.append((kind, rel, schema, name))
    if dry_run:
        return rows + [{"path": rel, "action": "would delete"} for _, rel, _, _ in todo]
    if not todo:
        logger.info("Cleanup: nothing to delete in %s", info["lakehouse_name"])
        return rows

    bad = [r for r in verify_archive_backup(archive_root_uri, ts) if r["status"] != "OK"]
    if bad:
        raise RuntimeError(f"Backup {ts} failed verification for {len(bad)} tables. Nothing was deleted.")

    logger.warning("Cleanup of %s: deleting %d items (mode %s)", info["lakehouse_name"], len(todo), mode)
    for kind, rel, schema, name in todo:
        path = f"{target_root}/{rel}"
        if kind == "table":
            identifier = (f"{catalog}.{q(schema)}.{q(name)}" if has_schemas
                          else f"{q(info['lakehouse_name'])}.{q(name)}")
            try:
                spark.sql(f"DROP TABLE IF EXISTS {identifier}")
            except Exception as exc:
                logger.info("DROP TABLE %s did not run cleanly: %s", identifier, reason(exc))
        if fs.exists(path):
            fs.rm(path, True)
        if fs.exists(path):
            raise RuntimeError(f"Could not delete {rel}. Cleanup stopped; the restore was not started.")
        rows.append({"path": rel, "action": "deleted"})
        logger.warning("Deleted %s", rel)
    return rows

<div style="border-top:1px solid rgba(127,127,127,0.4);margin-top:18px;padding-top:8px;font-size:12px;opacity:0.8">Backup and Restore Engine &middot; used through the Backup Utility and Restore Utility notebooks. The archive is never modified after a backup is written.</div>